In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import gmean

In [2]:
suites = pd.read_csv("rollup.by_suite.csv").rename(columns=lambda c: c.replace("sys.soc_0.core_0.", ""))
workloads = pd.read_csv("rollup.by_workload.csv").rename(columns=lambda c: c.replace("sys.soc_0.core_0.", ""))
traces = pd.read_csv("rollup.by_trace.csv").rename(columns=lambda c: c.replace("sys.soc_0.core_0.", ""))

In [3]:
si = suites.set_index(["configuration", "suite"]).sort_index()
wi = workloads.set_index(["configuration", "workload", "suite"]).sort_index()
ti = traces.set_index(["configuration", "workload", "trace", "suite"]).sort_index()

In [4]:
s_speedup = si.loc[:, "ipc"] / si.loc["baseline", "ipc"].rename("ipc speedup")
s_relevant_speedup = s_speedup.drop("baseline", level="configuration")
w_speedup = wi.loc[:, "ipc"] / wi.loc["baseline", "ipc"].rename("ipc speedup")
w_relevant_speedup = w_speedup.drop("baseline", level="configuration")
t_speedup = ti.loc[:, "ipc"] / ti.loc["baseline", "ipc"].rename("ipc speedup")
t_relevant_speedup = t_speedup.drop("baseline", level="configuration")

In [9]:
## The IPC speedup is a geomean across all workloads, ignoring the weight column.
## The weight column was used by the rollup script to weight the traces within each workload;
## the weight values given for workload and suite are not meaningful.
speedup_per_config = w_relevant_speedup.groupby(level="configuration").agg(gmean)
speedup_per_config.reset_index()

In [12]:
# speedup DataFrame indexed by suite (do not reset_index)
df = s_speedup.unstack(level="configuration")  # index = suite (15 entries)

ax = df.plot(figsize=(12, 6))  # plot with wider figure
ax.set_xticks(np.arange(len(df.index)))  # explicit tick positions
ax.set_xticklabels(df.index, rotation=90, ha="right")  # vertical labels, right-aligned
ax.set_xlabel("suite")
ax.set_ylabel("speedup")
plt.tight_layout()
plt.show()
ax.figure.savefig("by_suite.png")

In [14]:
w_spdplt = (
    w_speedup.unstack(level="configuration")
    .reset_index(drop=True)
    .apply(lambda c: c.sort_values(ignore_index=True), axis=0)
    .plot(linestyle="", marker=".", markersize=2, title="Workloads")
)
w_spdplt.figure.savefig("by_workload.png")

In [13]:
t_spdplt = (
    t_speedup.unstack(level="configuration")
    .reset_index(drop=True)
    .apply(lambda c: c.sort_values(ignore_index=True), axis=0)
    .plot(linestyle="", marker=".", markersize=2, title="Traces")
)
t_spdplt.figure.savefig("by_trace.png")

In [23]:
from IPython.display import HTML, display


def show_extremes(df, n=2):
    configs = pd.Index(df.index.get_level_values("configuration").unique()).sort_values()
    for cfg in configs:
        sel = df.loc[cfg]
        sorted_desc = sel.sort_values(ascending=True)
        df_disp = sorted_desc.reset_index()
        df_disp.insert(0, "configuration", cfg)
        html = df_disp.to_html(index=False, float_format="%.4f", max_rows=2 * n)
        display(HTML(html))

In [23]:
show_extremes(t_relevant_speedup, 2)